# EPA Station Monitoring + Climate Data Merge

**Goal:** Merge EPA water quality measurements with ISU climate data.

**Join keys:**
- **Location:** Each EPA monitoring location (lat/lon) is spatially matched to the nearest ISU climate station using a haversine nearest-neighbor search.
- **Time:** `ActivityStartDate` (EPA WQ) == `day` (climate).

**Input files:**
- `data/tabular/water-quality/raw/epa-wq.csv`
- `data/tabular/water-quality/raw/epa-stations.csv`
- `data/tabular/climate/raw/isu-climate.csv`

**Output:** `data/tabular/merged/epa-climate-merged.csv`

In [ ]:
import numpy as np
import pandas as pd
import requests
import json
import os
from scipy.spatial import cKDTree

## Step 1: Load and Clean EPA Water Quality Data

In [ ]:
wq_path = '../../../../data/tabular/water-quality/raw/epa-wq.csv'
df_wq = pd.read_csv(wq_path, low_memory=False)
print(f'epa-wq shape: {df_wq.shape}')

# Select key columns
wq_cols = [
    'ActivityIdentifier',
    'ActivityStartDate',
    'MonitoringLocationIdentifier',
    'MonitoringLocationName',
    'CharacteristicName',
    'ResultMeasureValue',
    'ResultMeasure/MeasureUnitCode',
    'ResultValueTypeName',
]
wq_cols = [c for c in wq_cols if c in df_wq.columns]
df_wq = df_wq[wq_cols].copy()

# Fix types
df_wq['ActivityStartDate'] = pd.to_datetime(df_wq['ActivityStartDate'], errors='coerce')
df_wq['ResultMeasureValue'] = pd.to_numeric(df_wq['ResultMeasureValue'], errors='coerce')

# Drop rows missing key fields
df_wq.dropna(subset=['MonitoringLocationIdentifier', 'ActivityStartDate', 'CharacteristicName', 'ResultMeasureValue'], inplace=True)

print(f'epa-wq cleaned shape: {df_wq.shape}')
df_wq.head(3)

## Step 2: Load and Clean EPA Stations Data

In [ ]:
stations_path = '../../../../data/tabular/water-quality/raw/epa-stations.csv'
df_stations = pd.read_csv(stations_path, low_memory=False)
print(f'epa-stations shape: {df_stations.shape}')

# Select key columns
station_cols = [
    'MonitoringLocationIdentifier',
    'MonitoringLocationName',
    'MonitoringLocationTypeName',
    'HUCEightDigitCode',
    'LatitudeMeasure',
    'LongitudeMeasure',
    'StateCode',
    'CountyCode',
]
station_cols = [c for c in station_cols if c in df_stations.columns]
df_stations = df_stations[station_cols].copy()

# Fix types
df_stations['LatitudeMeasure'] = pd.to_numeric(df_stations['LatitudeMeasure'], errors='coerce')
df_stations['LongitudeMeasure'] = pd.to_numeric(df_stations['LongitudeMeasure'], errors='coerce')

# Drop stations without coordinates
df_stations.dropna(subset=['LatitudeMeasure', 'LongitudeMeasure'], inplace=True)
df_stations.drop_duplicates(subset='MonitoringLocationIdentifier', inplace=True)

print(f'epa-stations cleaned shape: {df_stations.shape}')
df_stations.head(3)

## Step 3: Merge EPA WQ + EPA Stations on MonitoringLocationIdentifier

In [ ]:
df_epa = df_wq.merge(df_stations, on='MonitoringLocationIdentifier', how='left', suffixes=('', '_station'))

# Resolve MonitoringLocationName conflict (prefer WQ value, fall back to station)
if 'MonitoringLocationName_station' in df_epa.columns:
    df_epa['MonitoringLocationName'] = df_epa['MonitoringLocationName'].fillna(df_epa['MonitoringLocationName_station'])
    df_epa.drop(columns=['MonitoringLocationName_station'], inplace=True)

print(f'EPA merged shape: {df_epa.shape}')
print(f'Rows with lat/lon: {df_epa["LatitudeMeasure"].notna().sum():,}')
df_epa.head(3)

## Step 4: Fetch ISU Climate Station Coordinates from IEM

ISU climate stations are identified by codes (e.g. `IA1533`). We fetch their lat/lon from the IEM GeoJSON endpoint and filter to only stations present in our climate dataset.

In [ ]:
climate_path = '../../../../data/tabular/climate/raw/isu-climate.csv'
df_climate = pd.read_csv(climate_path)
print(f'Climate shape: {df_climate.shape}')
print(f'Unique climate stations: {df_climate["station"].nunique()}')

# Fix date type
df_climate['day'] = pd.to_datetime(df_climate['day'], errors='coerce')
df_climate.dropna(subset=['day'], inplace=True)

climate_stations_in_data = set(df_climate['station'].unique())
print(f'Date range: {df_climate["day"].min().date()} to {df_climate["day"].max().date()}')

In [ ]:
# Fetch station metadata from IEM
IEM_URL = 'https://mesonet.agron.iastate.edu/geojson/network/IACLIMATE.geojson'

response = requests.get(IEM_URL, timeout=30)
response.raise_for_status()
geojson = response.json()

climate_meta_rows = []
for feature in geojson['features']:
    sid = feature['id']
    lon, lat = feature['geometry']['coordinates']
    name = feature['properties'].get('sname', '')
    climate_meta_rows.append({'climate_station': sid, 'climate_station_name': name, 'climate_lat': lat, 'climate_lon': lon})

df_climate_meta = pd.DataFrame(climate_meta_rows)

# Keep only stations present in the climate dataset
df_climate_meta = df_climate_meta[df_climate_meta['climate_station'].isin(climate_stations_in_data)].reset_index(drop=True)

print(f'Climate stations with coordinates: {len(df_climate_meta)}')
df_climate_meta.head(5)

## Step 5: Spatial Nearest-Neighbor — Map Each EPA Station to Closest Climate Station

We use `scipy.spatial.cKDTree` with haversine-projected coordinates to find the nearest climate station for each EPA monitoring location.

In [ ]:
def haversine_deg_to_rad(lat_lon_deg):
    """Convert (lat, lon) degrees to radians scaled for haversine."""
    return np.radians(lat_lon_deg)

# Build KD-tree from climate station coordinates (in radians for haversine approx)
climate_coords_rad = np.radians(df_climate_meta[['climate_lat', 'climate_lon']].values)
tree = cKDTree(climate_coords_rad)

# Get unique EPA locations with lat/lon
df_epa_locs = df_epa[['MonitoringLocationIdentifier', 'LatitudeMeasure', 'LongitudeMeasure']].dropna().drop_duplicates()
print(f'Unique EPA locations with coordinates: {len(df_epa_locs)}')

epa_coords_rad = np.radians(df_epa_locs[['LatitudeMeasure', 'LongitudeMeasure']].values)

# Query nearest climate station for each EPA location
distances, indices = tree.query(epa_coords_rad, k=1)

# Earth radius ≈ 6371 km; convert radian distance to km
distances_km = distances * 6371

df_epa_locs = df_epa_locs.copy()
df_epa_locs['climate_station'] = df_climate_meta['climate_station'].iloc[indices].values
df_epa_locs['climate_station_name'] = df_climate_meta['climate_station_name'].iloc[indices].values
df_epa_locs['distance_to_climate_station_km'] = distances_km

print(f'\nNearest-neighbor distance stats (km):')
print(df_epa_locs['distance_to_climate_station_km'].describe().round(2))
df_epa_locs.head(5)

## Step 6: Join Climate Station Mapping Back to EPA Data

In [ ]:
# Merge climate station assignment into main EPA dataframe
df_epa = df_epa.merge(
    df_epa_locs[['MonitoringLocationIdentifier', 'climate_station', 'climate_station_name', 'distance_to_climate_station_km']],
    on='MonitoringLocationIdentifier',
    how='left'
)

print(f'EPA data with climate station assigned: {df_epa["climate_station"].notna().sum():,} / {len(df_epa):,} rows')
df_epa[['MonitoringLocationIdentifier', 'ActivityStartDate', 'climate_station', 'climate_station_name']].head(5)

## Step 7: Merge EPA Data with Climate Data on (climate_station, date)

In [ ]:
# Normalize date columns to date-only (no time component) for joining
df_epa['merge_date'] = df_epa['ActivityStartDate'].dt.normalize()
df_climate['merge_date'] = df_climate['day'].dt.normalize()

# Rename climate station column to match
df_climate_merge = df_climate.rename(columns={'station': 'climate_station'})

# Select climate columns to bring in
climate_feature_cols = ['climate_station', 'merge_date', 'doy', 'gdd_40_86', 'high', 'highc', 'low', 'lowc', 'precip', 'snow', 'snowd']
climate_feature_cols = [c for c in climate_feature_cols if c in df_climate_merge.columns]
df_climate_features = df_climate_merge[climate_feature_cols].drop_duplicates(subset=['climate_station', 'merge_date'])

# Merge on (climate_station, merge_date)
df_merged = df_epa.merge(
    df_climate_features,
    on=['climate_station', 'merge_date'],
    how='left'
)

# Drop helper column
df_merged.drop(columns=['merge_date'], inplace=True)

print(f'Final merged shape: {df_merged.shape}')
climate_match_rate = df_merged['high'].notna().sum() / len(df_merged) * 100
print(f'Climate match rate: {climate_match_rate:.1f}%')
df_merged.head(3)

## Step 8: Summary and Save

In [ ]:
print('=== Merged Dataset Summary ===')
print(f'Total rows:              {len(df_merged):,}')
print(f'Unique EPA locations:    {df_merged["MonitoringLocationIdentifier"].nunique():,}')
print(f'Unique climate stations: {df_merged["climate_station"].nunique():,}')
print(f'Date range:              {df_merged["ActivityStartDate"].min().date()} to {df_merged["ActivityStartDate"].max().date()}')
print(f'Climate match rate:      {climate_match_rate:.1f}%')
print(f'\nColumns: {df_merged.columns.tolist()}')
print(f'\nMissing values:')
print(df_merged.isnull().sum()[df_merged.isnull().sum() > 0])

In [ ]:
out_path = '../../../../data/tabular/merged'
os.makedirs(out_path, exist_ok=True)
out_file = f'{out_path}/epa-climate-merged.csv'
df_merged.to_csv(out_file, index=False)
print(f'Saved {len(df_merged):,} rows → {out_file}')

## (Optional) Station-to-Station Mapping Reference

In [ ]:
# Save the EPA → climate station mapping for reference
df_epa_locs[['MonitoringLocationIdentifier', 'LatitudeMeasure', 'LongitudeMeasure',
              'climate_station', 'climate_station_name', 'distance_to_climate_station_km']].to_csv(
    f'{out_path}/epa-to-climate-station-map.csv', index=False
)
print(f'Saved station mapping → {out_path}/epa-to-climate-station-map.csv')
df_epa_locs.sort_values('distance_to_climate_station_km').head(10)